# Fine-tuning of a pretrained foundation model
Using the ESOL dataset, we have seen that the linear-probe approach of a foundation model with a small regressor on top is about comparable with small regression models trained on Morgan fingerprints, but performs worse than the small models, when they are trained on molecular descriptors.

Fine-tuning makes the model more adaptive towards the target task. In this exercise, you will use basically the same architecture (`ChemBERTa-zinc-base-v1` + regressor (this time with one hidden layer)), but instead of using the pretrained model as a fixed encoder only, it is (partially) fine-tuned on the ESOL data. 

Since fine-tuning takes quite a bit of time, the notebook has been prerun and the final state of the model was saved for another transfer learning task (7B_TransferLearning).

## Your task in this notebook: 
Go through the code and try to understand the essential sections. Imagine now you are building a model for your team and you want your team members to understand the code as well - hence, comments are essentials. **Write brief comments in lines with 3 hashtags (###)** (sections 2-9).


1) Import dependencies

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np


2) Prepare the data: Import the dataset, train-test split, define the dataset class

In [ ]:
df_full = pd.read_csv("esol.csv")


df_full.dropna(axis=0, inplace=True) ### drop rows just in case either smiles or logS values are missing. 
# It is crucial to have complete and labelled data for our exercise!


df = df_full
# df = df_full.sample(sample_size, random_state=42) # use either the full dataset or a much smaller one by sampling
print(df.head())
print(f"Dataset size: {len(df)}")

                                              smiles  logS
0  OCC3OC(OCC2OC(OC(C#N)c1ccccc1)C(O)C(O)C2O)C(O)... -0.77
1                             Cc1occc1C(=O)Nc2ccccc2 -3.30
2                               CC(C)=CCCC(C)=CC(=O) -2.06
3                 c1ccc2c(c1)ccc3c2ccc4c5ccccc5ccc43 -7.87
4                                            c1ccsc1 -1.33
Dataset size: 1128


In [ ]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42 ### ensures reproducibility, will always give the same random split
)

In [ ]:
class ESOLDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128): ### function that will be carried out when the Dataset class is called for the first time
        self.df = df
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        smiles = self.df.iloc[idx]["smiles"]
        label = self.df.iloc[idx]["logS"]

        enc = self.tokenizer(
            smiles,
            truncation=True,
            padding="max_length", ### makes sure that length is the same for all
            max_length=self.max_length,
            return_tensors="pt" ###returns output as pytorch tensors
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0), ### returns the tokenised sequence as a 1D tensor reducing its dimension to (maxlength)
            "attention_mask": enc["attention_mask"].squeeze(0), ### tells encoder which tokens are real and which ones should be ignored (e.g. padding)
            "labels": torch.tensor(label, dtype=torch.float)
        }


3) Define the architecture:

In [ ]:
class chemberta_esol_regressor(nn.Module):
    def __init__(self, model_name, hidden_dim=None):
        super().__init__()

        # general encoder
        self.encoder = AutoModel.from_pretrained(model_name) ### encoder = foundation model

        if hidden_dim is None:
            hidden_dim = self.encoder.config.hidden_size ### read hidden_size from foundation model unless specified

        # Regression head (task-specific)
        self.fc1 = nn.Linear(hidden_dim, 256) ### one hidden layer
        self.act = nn.ReLU() ### Activation function (returns x for x>0 and 0 for x<=0)
        self.dropout = nn.Dropout(p=0.2) 
        self.fc2 = nn.Linear(256, 1) ### output layer

    def forward(self, input_ids, attention_mask): ### defines the forward pass
        # Encoder forward
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Pooling (explicit & visible)
        # Use [CLS] token representation
        cls_embedding = outputs.last_hidden_state[:, 0, :] 

        # Regression head forward
        x = self.fc1(cls_embedding)
        x = self.act(x)
        x = self.dropout(x)
        x = self.fc2(x)

        return x.squeeze(-1)


4) Load foundation model

In [ ]:
MODEL_NAME = "seyonec/ChemBERTa-zinc-base-v1" ### foundation model

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME) ### uses the tokeniser from the foundation model
model = chemberta_esol_regressor(MODEL_NAME)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: seyonec/ChemBERTa-zinc-base-v1
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


5) Define model parameters

In [ ]:
BATCH_SIZE = 32 ### batch size for training
EPOCHS = 20

### Different learning rates for the head and the encoder (to prevent catastrophic forgetting)
LR_ENCODER = 1e-5
LR_HEAD = 1e-3

N_UNFROZEN_LAYERS = 2  ### partial fine-tuning

6) Partial fine-tuning

6.1. Freeze everything:

In [ ]:
for param in model.encoder.parameters(): # "encoder" here is the model name defined in our NN architecture
    param.requires_grad = False ### nothing is retrained


6.2. Unfreeze last k transformer layers of foundation model

In [ ]:
encoder_layers = model.encoder.encoder.layer # the first "encoder" is the name defined in our model architecture,
# the second one is the location, where the layers are stored in the ChemBERTa model, e.g. model.chemberta.encoder.layer

for layer in encoder_layers[-N_UNFROZEN_LAYERS:]: ### slice of the end of a layer list
    for param in layer.parameters():
        param.requires_grad = True ### parameters in the unfrozen layers require gradient = will be retrained

6.3. Optionally: Unfreeze / retrain final Layer Normalisation of the encoder

In [93]:
for param in model.encoder.embeddings.LayerNorm.parameters(): # "encoder" here is the model name defined in our NN architecture
    param.requires_grad = True


7) Initiate DataLoaders

In [ ]:
train_dataset = ESOLDataset(train_df, tokenizer)
val_dataset   = ESOLDataset(val_df, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True) ### loader = wrapper for dataset - distributes it into batches, shuffling randomises the smaller batch arrays
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE) ### shuffling not needed for evaluation


8) Define Optimizer with different learning rates for the encoder and the regressor model

In [ ]:
optimizer = AdamW(
    [
        {
            "params": encoder_layers[-N_UNFROZEN_LAYERS:].parameters(), ### specifies for which layers the parameters are updated
            "lr": LR_ENCODER,
        },
        {
            "params": list(model.fc1.parameters()) + list(model.fc2.parameters()),
            "lr": LR_HEAD,
        },
    ],
    weight_decay=1e-2 ### regularisation parameter
)

loss_fn = nn.MSELoss()



9) Define evaluation function and initiate training.

In [ ]:
def evaluate(model, dataloader):
    model.eval() ### evaluation mode turns off the gradient and all training-related functions (e.g. dropout)
    preds, targets = [], []

    with torch.no_grad(): ### no gradient = params not touched
        for batch in dataloader:
            input_ids = batch["input_ids"]
            attention_mask = batch["attention_mask"] ### see above (real tokens vs. tokens to ignore)
            labels = batch["labels"]

            outputs = model(input_ids, attention_mask)
            preds.append(outputs.cpu().numpy())
            targets.append(labels.cpu().numpy())

    preds = np.concatenate(preds)
    targets = np.concatenate(targets)
    rmse = np.sqrt(mean_squared_error(targets, preds))
    return rmse

In [ ]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels = batch["labels"]

        outputs = model(input_ids, attention_mask)
        loss = loss_fn(outputs, labels)

        loss.backward() ### backpropagation
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() ### loss needs to be accumulated over all batches and then averaged 

    train_loss = total_loss / len(train_loader)
    val_rmse = evaluate(model, val_loader)

    print(
        f"Epoch {epoch+1:02d} | "
        f"Train RMSE: {np.sqrt(train_loss):.4f} | "
        f"Val RMSE: {val_rmse:.4f}"
    )

Epoch 01 | Train RMSE: 1.8571 | Val RMSE: 1.4460
Epoch 02 | Train RMSE: 1.3487 | Val RMSE: 1.2706
Epoch 03 | Train RMSE: 1.3036 | Val RMSE: 1.2083
Epoch 04 | Train RMSE: 1.2642 | Val RMSE: 1.1699
Epoch 05 | Train RMSE: 1.2236 | Val RMSE: 1.1321
Epoch 06 | Train RMSE: 1.1513 | Val RMSE: 1.1785
Epoch 07 | Train RMSE: 1.1717 | Val RMSE: 1.1286
Epoch 08 | Train RMSE: 1.1120 | Val RMSE: 1.1339
Epoch 09 | Train RMSE: 1.1404 | Val RMSE: 1.1343
Epoch 10 | Train RMSE: 1.1188 | Val RMSE: 1.0503
Epoch 11 | Train RMSE: 1.0676 | Val RMSE: 1.0761
Epoch 12 | Train RMSE: 1.0768 | Val RMSE: 1.0571
Epoch 13 | Train RMSE: 1.0284 | Val RMSE: 1.0475
Epoch 14 | Train RMSE: 1.0059 | Val RMSE: 1.0817
Epoch 15 | Train RMSE: 1.0546 | Val RMSE: 1.0856
Epoch 16 | Train RMSE: 0.9876 | Val RMSE: 1.0743
Epoch 17 | Train RMSE: 0.9964 | Val RMSE: 1.0083
Epoch 18 | Train RMSE: 0.9838 | Val RMSE: 1.0706
Epoch 19 | Train RMSE: 0.9717 | Val RMSE: 1.1473
Epoch 20 | Train RMSE: 0.9481 | Val RMSE: 1.1721


10) Save the task-specific encoder as pretrained model - Hugging Face (HF) style (this can be reloaded like any HF model - see snippet in assignment 7B). Note: This is not the same as saving the entire model via pytorch (`torch.save(model)`) - the solution below is ergo not unstable and brittle, but highly reusable for encodings!

In [98]:
model.encoder.save_pretrained("chemberta_esol_encoder")
tokenizer.save_pretrained("chemberta_esol_encoder")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('chemberta_esol_encoder\\tokenizer_config.json',
 'chemberta_esol_encoder\\tokenizer.json')

11) Save the state of the entire model (encoder and regressor) to reload the pretrained regressor as well later on (could be done separately as well).

In [101]:
torch.save(model.state_dict(), "chemberta_esol_regressor.pt")


Note that the model architecture is not stored in any of these state dicts - either move it to a separate file (e.g. `models.py`), or copy-paste the class definition wherever you need it again - simply strip it down to the bare architecture (see 7B).